In [4]:
import pandas as pd
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

In [5]:
df = pd.read_excel(r"..\..\data\Processed_Data\Clustered_Data.xlsx")

In [6]:
FEATURE_GLOSSARY = {
    # Sector Values Added (% of GDP)
    "agriculture_forestry_and_fishing_value_added_of_gdp": (
        "Share of GDP generated by agriculture, forestry, and fishing sectors. Higher values indicate heavy economic reliance on primary sectors."
    ),
    "industry_including_construction_value_added_of_gdp": (
        "Share of GDP generated by industrial production, manufacturing, mining, and construction sectors."
    ),
    "services_value_added_of_gdp": (
        "Share of GDP generated by the service sector (wholesale, retail, finance, healthcare, and education)."
    ),

    # Environmental & Energy Metrics
    "carbon_intensity_of_gdp_kg_co2e_per_constant_2015_us_of_gdp": (
        "Carbon dioxide emissions produced per unit of constant 2015 US dollar GDP. Measures economic carbon efficiency."
    ),
    "forest_area_of_land_area": (
        "Percentage of total land area covered by natural or planted forests."
    ),
    "renewable_electricity_output_of_total_electricity_output": (
        "Share of total electricity output generated from renewable resources (hydro, solar, wind, geothermal)."
    ),
    "renewable_energy_consumption_of_total_final_energy_consumption": (
        "Share of total final energy consumption derived from renewable energy sources."
    ),
    "Energy_Environmental_Index": (
        "Composite index measuring overall environmental sustainability, clean energy output, and carbon footprint efficiency."
    ),

    # Demographic & Social Indicators
    "fertility_rate_total_births_per_woman": (
        "Average number of children that would be born to a woman over her childbearing years."
    ),
    "population_growth_annual_": (
        "Annual exponential population growth rate expressed as a percentage."
    ),
    "school_enrollment_primary_gross": (
        "Gross enrollment ratio in primary education regardless of official age group, relative to total population of official primary school age."
    ),
    "current_health_expenditure_of_gdp": (
        "Total healthcare expenditure expressed as a percentage of overall national GDP."
    ),

    # Economic Growth, Employment & Indices
    "gdp_growth_annual_": (
        "Annual percentage growth rate of GDP at market prices based on constant local currency."
    ),
    "unemployment_total_of_total_labor_force_modeled_ilo_estimate": (
        "Percentage of total labor force that is without work but available and seeking employment (ILO modeled estimate)."
    ),
    "Socioeconomic_developmen_Index": (
        "Composite index combining economic performance, human capital, healthcare access, and quality of life indicators."
    ),

    # Log-Transformed Features (Used in Clustering Pipeline)
    "fdi_log": (
        "Log-transformed Foreign Direct Investment (net inflows) to correct skewness."
    ),
    "inflation_log": (
        "Log-transformed annual consumer price inflation percentage to control for extreme outlier rates."
    ),
    "gdp_per_capita_constant_2015_us_log": (
        "Log-transformed GDP per capita in constant 2015 USD, reflecting standard of living."
    ),

    # Original Unscaled Values
    "fdi_original": (
        "Net Foreign Direct Investment inflows expressed as a percentage of GDP in raw format."
    ),
    "gdp_per_capita_original": (
        "Gross Domestic Product per capita in constant 2015 US dollars in raw format."
    ),
    "inflation_original": (
        "Annual percentage change in consumer price index (inflation rate) in raw format."
    ),
    
    # Clustering Categorization
    "Cluster": (
        "Assigned machine learning cluster grouping countries with similar economic and socio-environmental profiles."
    )
}

In [7]:
def generate_country_profiles_with_cluster_metrics(
    df_path: str,
    scaler_path: str,
    pca_path: str,
    kmeans_path: str,
    feature_cols: list
) -> list[dict]:
    """
    Loads clustered data along with ML model artifacts to construct 
    detailed country profile documents for RAG indexing.
    """
    # 1. Load Data and Artifacts
    df = pd.read_excel(df_path)
    scaler = joblib.load(scaler_path)
    pca = joblib.load(pca_path)
    kmeans = joblib.load(kmeans_path)

    # 2. Extract features & calculate PCA coordinates and centroid distances
    X_raw = df[feature_cols].values
    X_scaled = scaler.transform(X_raw)
    X_pca = pca.transform(X_scaled)
    
    # Calculate distance to each country's assigned cluster centroid in PCA space
    centroids = kmeans.cluster_centers_
    cluster_labels = df['Cluster'].values
    
    # Ensure cluster labels match centroid indices (e.g., if labels are string "Cluster_1", map to 0, 1, ...)
    if isinstance(cluster_labels[0], str):
        cluster_indices = [int(str(c).split('_')[-1]) - 1 for c in cluster_labels]
    else:
        cluster_indices = [int(c) for c in cluster_labels]

    distances = []
    for i, cluster_idx in enumerate(cluster_indices):
        dist = np.linalg.norm(X_pca[i] - centroids[cluster_idx])
        distances.append(dist)

    df['centroid_distance'] = distances
    for dim in range(X_pca.shape[1]):
        df[f'PCA_Component_{dim+1}'] = X_pca[:, dim]

    # 3. Generate Documents
    documents = []
    metadata_exclude = ['country_name', 'country_code', 'Cluster', 'centroid_distance']

    for idx, row in df.iterrows():
        country_name = row['country_name']
        country_code = row['country_code']
        cluster_id = row['Cluster']
        dist = row['centroid_distance']

        # Format PCA coordinates
        pca_coords_str = ", ".join([
            f"PC{dim+1}: {row[f'PCA_Component_{dim+1}']:.4f}"
            for dim in range(X_pca.shape[1])
        ])

        # Format all variable metrics
        metrics_list = []
        for col in df.columns:
            if col not in metadata_exclude and not col.startswith('PCA_Component_'):
                val = row[col]
                formatted_val = f"{val:,.2f}" if isinstance(val, (int, float)) else str(val)
                metrics_list.append(f"  - {col}: {formatted_val}")

        metrics_str = "\n".join(metrics_list)

        # Construct Full Document Text
        content = (
            f"Country Profile: {country_name} ({country_code})\n"
            f"Assigned Cluster: {cluster_id}\n"
            f"Distance to Cluster Centroid: {dist:.4f}\n"
            f"PCA Coordinates: [{pca_coords_str}]\n\n"
            f"Socioeconomic & Environmental Features:\n"
            f"{metrics_str}\n"
        )

        documents.append({
            "page_content": content,
            "metadata": {
                "doc_type": "country_profile",
                "country_name": country_name,
                "country_code": country_code,
                "cluster": str(cluster_id),
                "centroid_distance": float(dist)
            }
        })

    return documents

In [8]:
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "processing" else Path.cwd()

# Define feature list used in fitting scaler & PCA
features = [
    'agriculture_forestry_and_fishing_value_added_of_gdp',
    'carbon_intensity_of_gdp_kg_co2e_per_constant_2015_us_of_gdp',
    'current_health_expenditure_of_gdp',
    'fertility_rate_total_births_per_woman',
    'forest_area_of_land_area',
    'gdp_growth_annual_',
    'industry_including_construction_value_added_of_gdp',
    'population_growth_annual_',
    'renewable_electricity_output_of_total_electricity_output',
    'renewable_energy_consumption_of_total_final_energy_consumption',
    'school_enrollment_primary_gross',
    'services_value_added_of_gdp',
    'unemployment_total_of_total_labor_force_modeled_ilo_estimate',
    'Socioeconomic_developmen_Index',
    'Energy_Environmental_Index',
    'fdi_log',
    'inflation_log',
    'gdp_per_capita_constant_2015_us_log'
]

country_docs = generate_country_profiles_with_cluster_metrics(
    df_path=PROJECT_ROOT / "data" / "Processed_Data" / "Clustered_Data.xlsx",
    scaler_path=PROJECT_ROOT / "notebooks" / "scaler.pkl",
    pca_path=PROJECT_ROOT / "notebooks" / "pca.pkl",
    kmeans_path=PROJECT_ROOT / "notebooks" / "kmeans.pkl",
    feature_cols=features
)

# Inspect first document output
print(country_docs[0]['page_content'])

Country Profile: Saudi Arabia (SAU)
Assigned Cluster: Cluster_1
Distance to Cluster Centroid: 1.4525
PCA Coordinates: [PC1: 1.7714, PC2: 3.2488]

Socioeconomic & Environmental Features:
  - agriculture_forestry_and_fishing_value_added_of_gdp: 2.50
  - carbon_intensity_of_gdp_kg_co2e_per_constant_2015_us_of_gdp: 0.84
  - current_health_expenditure_of_gdp: 5.20
  - fertility_rate_total_births_per_woman: 2.61
  - forest_area_of_land_area: 0.45
  - gdp_growth_annual_: 3.72
  - industry_including_construction_value_added_of_gdp: 50.77
  - population_growth_annual_: 2.00
  - renewable_electricity_output_of_total_electricity_output: 0.03
  - renewable_energy_consumption_of_total_final_energy_consumption: 0.02
  - school_enrollment_primary_gross: 109.13
  - services_value_added_of_gdp: 44.91
  - unemployment_total_of_total_labor_force_modeled_ilo_estimate: 5.95
  - Socioeconomic_developmen_Index: 0.83
  - Energy_Environmental_Index: 1.82
  - fdi_log: 0.58
  - inflation_log: 1.04
  - gdp_per_ca

In [9]:
def generate_cluster_summary_documents(
    df: pd.DataFrame,
    feature_cols: list,
    cluster_col: str = 'Cluster',
    cluster_labels_map: dict = None
) -> list[dict]:
    """
    Computes full descriptive statistics (.describe()) per cluster and formats
    comprehensive Cluster Summary documents for RAG indexing.
    """
    cluster_docs = []
    
    # Map default labels if none provided
    if cluster_labels_map is None:
        unique_clusters = sorted(df[cluster_col].unique())
        cluster_labels_map = {c: f"Cluster {c}" for c in unique_clusters}

    for cluster_id, group in df.groupby(cluster_col):
        cluster_name = cluster_labels_map.get(cluster_id, f"Cluster {cluster_id}")
        cluster_size = len(group)
        pct_of_total = (cluster_size / len(df)) * 100
        member_countries = sorted(group['country_name'].tolist())

        # Run describe() on numerical feature columns for this cluster
        stats_df = group[feature_cols].describe().T  # transpose to feature-wise rows
        
        # Build detailed statistical summary text
        stats_lines = []
        for feat, row in stats_df.iterrows():
            stats_lines.append(
                f"  - {feat}:\n"
                f"      Mean: {row['mean']:.2f} | Std: {row['std']:.2f}\n"
                f"      Median (50%): {row['50%']:.2f} | Min: {row['min']:.2f} | Max: {row['max']:.2f}"
            )
        stats_text = "\n".join(stats_lines)

        # Identify top key traits relative to overall global averages
        global_means = df[feature_cols].mean()
        cluster_means = stats_df['mean']
        relative_diff = ((cluster_means - global_means) / global_means) * 100
        
        top_high = relative_diff.sort_values(ascending=False).head(3)
        top_low = relative_diff.sort_values(ascending=True).head(3)
        
        distinguishing_traits = (
            "Key Distinguishing Characteristics (Relative to Global Mean):\n"
            + "\n".join([f"  - Significantly Higher: {feat} (+{val:.1f}%)" for feat, val in top_high.items()]) + "\n"
            + "\n".join([f"  - Significantly Lower: {feat} ({val:.1f}%)" for feat, val in top_low.items()])
        )

        # Build Full Structured Document Text
        content = (
            f"Cluster Summary Profile: {cluster_name}\n"
            f"Cluster ID: {cluster_id}\n"
            f"Size: {cluster_size} Countries ({pct_of_total:.1f}% of global dataset)\n\n"
            f"Member Countries:\n{', '.join(member_countries)}\n\n"
            f"{distinguishing_traits}\n\n"
            f"Detailed Statistical Metrics (.describe()):\n"
            f"{stats_text}\n"
        )

        cluster_docs.append({
            "page_content": content,
            "metadata": {
                "doc_type": "cluster_summary",
                "cluster_id": str(cluster_id),
                "cluster_name": cluster_name,
                "country_count": cluster_size
            }
        })

    return cluster_docs

In [10]:
# Define custom cluster archetype labels if you have them (Optional)
cluster_labels = {
    "Cluster_1": "Cluster 1: High-Growth Emerging Market Economies",
    "Cluster_2": "Cluster 2: Advanced High-Income & Service-Driven Economies",
    "Cluster_3": "Cluster 3: Low-Income Agriculture & Resource Dependent Nations"
}

# Generate cluster documents from loaded dataframe `df`
cluster_documents = generate_cluster_summary_documents(
    df=df,
    feature_cols=features,
    cluster_col='Cluster',
    cluster_labels_map=cluster_labels
)

# Print first summary document preview
print(cluster_documents[0]["page_content"])

Cluster Summary Profile: Cluster 1: High-Growth Emerging Market Economies
Cluster ID: Cluster_1
Size: 39 Countries (21.4% of global dataset)

Member Countries:
Algeria, Azerbaijan, Bahrain, Belarus, Bolivia, Botswana, Brunei Darussalam, China, Egypt, Arab Rep., Equatorial Guinea, Indonesia, Iran, Islamic Rep., Iraq, Jordan, Kazakhstan, Kuwait, Libya, Malaysia, Moldova, Mongolia, Morocco, Oman, Qatar, Russian Federation, Saudi Arabia, South Africa, Suriname, Syrian Arab Republic, Thailand, Trinidad and Tobago, Tunisia, Turkiye, Turkmenistan, Ukraine, United Arab Emirates, Uzbekistan, Venezuela, RB, Viet Nam, Yemen, Rep.

Key Distinguishing Characteristics (Relative to Global Mean):
  - Significantly Higher: carbon_intensity_of_gdp_kg_co2e_per_constant_2015_us_of_gdp (+105.5%)
  - Significantly Higher: industry_including_construction_value_added_of_gdp (+46.5%)
  - Significantly Higher: inflation_log (+22.2%)
  - Significantly Lower: Socioeconomic_developmen_Index (-256190218738702752.0%

In [11]:
def generate_feature_glossary_document(
    df: pd.DataFrame,
    feature_glossary: dict,
    feature_cols: list
) -> dict:
    """
    Generates a single comprehensive feature glossary document containing 
    definitions, domain meanings of high/low values, and dataset-wide benchmarks.
    """
    
    # Quantitative benchmark interpretations mapped to each feature
    FEATURE_INTERPRETATIONS = {
        "agriculture_forestry_and_fishing_value_added_of_gdp": {
            "high": "High reliance on agrarian economy, typical of developing/low-income nations.",
            "low": "Industrialized or service-based economy with low primary-sector dependency."
        },
        "industry_including_construction_value_added_of_gdp": {
            "high": "Strong manufacturing, construction, or resource-extraction industrial base.",
            "low": "Economy heavily dominated by services or primary agricultural activities."
        },
        "services_value_added_of_gdp": {
            "high": "Advanced service-oriented economy (finance, tech, retail, tourism).",
            "low": "Less developed tertiary sector; higher reliance on agriculture or industry."
        },
        "carbon_intensity_of_gdp_kg_co2e_per_constant_2015_us_of_gdp": {
            "high": "Carbon-heavy energy infrastructure; low environmental efficiency per GDP unit.",
            "low": "Clean, highly energy-efficient economy or service-heavy low-emission industry."
        },
        "forest_area_of_land_area": {
            "high": "Extensive forest coverage; strong natural carbon sink capacity.",
            "low": "Arid, highly urbanized, or heavily deforested/agricultural territory."
        },
        "renewable_electricity_output_of_total_electricity_output": {
            "high": "Strong clean grid transition (solar, hydro, wind, geothermal power).",
            "low": "Heavy reliance on fossil fuels (coal, natural gas, oil) for electricity generation."
        },
        "renewable_energy_consumption_of_total_final_energy_consumption": {
            "high": "High reliance on renewables across total energy consumption (heating, transport, power).",
            "low": "Low integration of clean energy across overall national energy consumption."
        },
        "fertility_rate_total_births_per_woman": {
            "high": "Rapid population expansion potential; typical of younger demographic profiles.",
            "low": "Aging population risk; birth rates near or below replacement level (2.1)."
        },
        "population_growth_annual_": {
            "high": "Fast annual demographic growth due to high birth rates or net immigration.",
            "low": "Stagnant or shrinking population; potential labor supply shortages."
        },
        "school_enrollment_primary_gross": {
            "high": "Broad primary education access across the child-age demographic.",
            "low": "Educational access barriers or structural underinvestment in early education."
        },
        "current_health_expenditure_of_gdp": {
            "high": "Substantial national investment in healthcare services and medical infrastructure.",
            "low": "Limited public/private healthcare spending relative to economic output."
        },
        "gdp_growth_annual_": {
            "high": "Rapidly expanding national economy; strong year-over-year production growth.",
            "low": "Economic stagnation, sluggish performance, or contraction."
        },
        "unemployment_total_of_total_labor_force_modeled_ilo_estimate": {
            "high": "Elevated labor underutilization; potential economic distress or job scarcity.",
            "low": "Strong labor demand and near full employment conditions."
        },
        "Socioeconomic_developmen_Index": {
            "high": "Strong socio-economic development, robust living standards, and public services.",
            "low": "Lower human capital index and persistent socio-economic challenges."
        },
        "Energy_Environmental_Index": {
            "high": "Superior eco-efficiency, clean energy adoption, and green infrastructure.",
            "low": "Severe environmental footprint or under-developed clean energy resources."
        },
        "fdi_log": {
            "high": "Strong foreign capital inflows, high international investor confidence.",
            "low": "Low foreign direct investment attraction relative to overall GDP."
        },
        "inflation_log": {
            "high": "Accelerated consumer price growth; potential currency de-valuation.",
            "low": "Stable consumer pricing environment or low inflation/deflation dynamics."
        },
        "gdp_per_capita_constant_2015_us_log": {
            "high": "High standard of living and substantial output per individual.",
            "low": "Low per-capita income baseline, lower average purchasing power."
        }
    }

    # Calculate dataset global summary statistics (.describe())
    stats_df = df[feature_cols].describe().T

    sections = []
    for feat in feature_cols:
        desc = feature_glossary.get(feat, "Socioeconomic or environmental indicator.")
        interp = FEATURE_INTERPRETATIONS.get(feat, {
            "high": "Above-average value for this economic dimension.",
            "low": "Below-average value for this economic dimension."
        })
        
        row = stats_df.loc[feat] if feat in stats_df.index else None
        
        if row is not None:
            stats_str = (
                f"    * Global Benchmarks: Min={row['min']:.2f} | 25%={row['25%']:.2f} | "
                f"Median={row['50%']:.2f} | Mean={row['mean']:.2f} | 75%={row['75%']:.2f} | Max={row['max']:.2f}"
            )
        else:
            stats_str = "    * Global Benchmarks: Not available"

        feat_entry = (
            f"Feature: `{feat}`\n"
            f"  - Description: {desc}\n"
            f"  - Value Significance:\n"
            f"    * High Value Signifies: {interp['high']}\n"
            f"    * Low Value Signifies: {interp['low']}\n"
            f"{stats_str}"
        )
        sections.append(feat_entry)

    content = (
        "FEATURE CONTEXT, DEFINITIONS & INTERPRETATION GUIDE\n\n"
        "This document provides exact definitions, global benchmarks, and qualitative interpretations "
        "for all variables present in the country clustering dataset. Use this reference to contextualize "
        "high or low feature values when evaluating country profiles and cluster archetypes.\n\n"
        + "\n\n".join(sections)
    )

    return {
        "page_content": content,
        "metadata": {
            "doc_type": "feature_glossary",
            "total_features": len(feature_cols)
        }
    }

In [12]:
# Pass your df, the FEATURE_GLOSSARY dictionary defined earlier, and your feature column list
glossary_doc = generate_feature_glossary_document(
    df=df,
    feature_glossary=FEATURE_GLOSSARY,
    feature_cols=features
)

# Preview the generated glossary content
print(glossary_doc["page_content"])

FEATURE CONTEXT, DEFINITIONS & INTERPRETATION GUIDE

This document provides exact definitions, global benchmarks, and qualitative interpretations for all variables present in the country clustering dataset. Use this reference to contextualize high or low feature values when evaluating country profiles and cluster archetypes.

Feature: `agriculture_forestry_and_fishing_value_added_of_gdp`
  - Description: Share of GDP generated by agriculture, forestry, and fishing sectors. Higher values indicate heavy economic reliance on primary sectors.
  - Value Significance:
    * High Value Signifies: High reliance on agrarian economy, typical of developing/low-income nations.
    * Low Value Signifies: Industrialized or service-based economy with low primary-sector dependency.
    * Global Benchmarks: Min=0.00 | 25%=2.36 | Median=7.05 | Mean=10.66 | 75%=16.93 | Max=37.44

Feature: `carbon_intensity_of_gdp_kg_co2e_per_constant_2015_us_of_gdp`
  - Description: Carbon dioxide emissions produced per 

In [13]:
def generate_pipeline_methodology_document() -> dict:
    """
    Generates a structured methodology document detailing the exact data cleaning,
    imputation, feature engineering, and clustering techniques used in this analysis.
    """
    
    content = (
        "DATA PIPELINE, PREPROCESSING & METHODOLOGY DOCUMENTATION\n\n"
        "1. RAW DATA LOADING & INITIAL FILTERING:\n"
        "   - Source Data: World Development Indicators (2010–2025 panel dataset).\n"
        "   - Placeholder Handling: Replaced string missing placeholders ('..') with np.nan.\n"
        "   - Temporal Filtering: Dropped years 2022 to 2025 due to excessive missingness (>50%+ missing values across all indicators).\n"
        "   - Entity Validation: Used `pycountry` ISO 3166-1 alpha-3 code verification to retain valid country entities only.\n\n"
        "2. HIGH MISSINGNESS & FEATURE DROPPING CRITERIA:\n"
        "   - Country Removal: Calculated total missingness percentage per country across 2010–2021. Removed entities in the top 15% quantile of missingness (nan_percentage >= 85th percentile).\n"
        "   - Indicator Removal: Evaluated overall feature null ratios; dropped indicators with >= 50% missing values (e.g., multidimensional poverty headcount ratio).\n"
        "   - Negative Value Correction: Replaced invalid negative values in percentage features (such as fossil fuel consumption and gross capital formation) with np.nan.\n"
        "   - Redundant Feature Removal: Dropped raw trade sub-components (`exports` and `imports`) to prioritize composite economic indicators.\n\n"
        "3. LOG TRANSFORMATIONS & FEATURE SKEWNESS REDUCTION:\n"
        "   - Log-transformed variables using natural log np.log1p(x) to normalize right-skewed distributions:\n"
        "     * GDP per Capita (`gdp_per_capita_constant_2015_us_log`)\n"
        "     * Broad Money (% of GDP)\n"
        "     * Trade (% of GDP)\n"
        "     * Domestic Credit to Private Sector (% of GDP)\n"
        "     * Military Expenditure (% of GDP)\n"
        "   - Signed Log Transformation applied to variables with negative or zero values (e.g., net FDI inflows, annual inflation):\n"
        "     `sign(x) * np.log1p(|x|)`\n\n"
        "4. COMPOSITE INDEX CONSTRUCTION:\n"
        "   - Socioeconomic Development Index: Constructed by standardizing (StandardScaler) and taking the mean of:\n"
        "     * Access to electricity (% of population)\n"
        "     * Individuals using the Internet (% of population)\n"
        "     * Life expectancy at birth (years)\n"
        "     * Urban population (% of total)\n"
        "   - Energy & Environmental Index: Constructed by standardizing (StandardScaler) log-transformed values of:\n"
        "     * Energy use per capita (kg of oil equivalent)\n"
        "     * Carbon dioxide emissions per capita (t CO2e)\n"
        "   - Note: Individual component columns were dropped after composite index calculation to prevent multicollinearity.\n\n"
        "5. MULTI-STAGE NULL IMPUTATION STRATEGY:\n"
        "   For remaining features with null values (e.g., primary school enrollment at 15.3%, renewable electricity at 3.1%, inflation at 3.1%):\n"
        "   - Step 1 (Linear Interpolation): Applied linear interpolation within each country (`groupby('country_code')`) with limit=3 (inside values only) to preserve historical trends.\n"
        "   - Step 2 (Country-level Median): Filled remaining missing values using the individual country's historical median.\n"
        "   - Step 3 (Global Annual Median): Imputed remaining nulls using the global annual median across all countries for that specific year.\n\n"
        "6. AGGREGATION, REDUCTION & CLUSTERING PIPELINE:\n"
        "   - Temporal Aggregation: Computed 2010–2021 country-level mean across all 18 final features, creating a single static row per country.\n"
        "   - Standardization: Normalized all 18 aggregated features using `StandardScaler` (zero mean, unit variance).\n"
        "   - Principal Component Analysis (PCA):\n"
        "     * Reduced 18 features to 2 principal components.\n"
        "     * Total Explained Variance: 46.46% (PC1 = 33.1% Variance - General Development/Economy; PC2 = 13.4% Variance - Energy/Emissions).\n"
        "   - K-Means Clustering Selection:\n"
        "     * Evaluated cluster candidates (k=2 to 8) using Elbow Method (Inertia) and Silhouette Scores.\n"
        "     * k=2: Silhouette = 0.2396 | k=3: Silhouette = 0.2104 | k=4: Silhouette = 0.1531\n"
        "     * Selected Model: K-Means with k=3 clusters fitted on PCA-transformed space.\n"
        "   - Model Validation Scores:\n"
        "     * Silhouette Score: 0.380 (on 2D PCA space)\n"
        "     * Davies-Bouldin Index: 0.812\n"
        "     * Calinski-Harabasz Index: 187.214\n"
    )

    return {
        "page_content": content,
        "metadata": {
            "doc_type": "methodology_pipeline",
            "total_features_used": 18,
            "pca_components": 2,
            "n_clusters": 3
        }
    }

In [14]:
# Generate methodology document
methodology_doc = generate_pipeline_methodology_document()

print(methodology_doc["page_content"])

DATA PIPELINE, PREPROCESSING & METHODOLOGY DOCUMENTATION

1. RAW DATA LOADING & INITIAL FILTERING:
   - Source Data: World Development Indicators (2010–2025 panel dataset).
   - Placeholder Handling: Replaced string missing placeholders ('..') with np.nan.
   - Temporal Filtering: Dropped years 2022 to 2025 due to excessive missingness (>50%+ missing values across all indicators).
   - Entity Validation: Used `pycountry` ISO 3166-1 alpha-3 code verification to retain valid country entities only.

2. HIGH MISSINGNESS & FEATURE DROPPING CRITERIA:
   - Country Removal: Calculated total missingness percentage per country across 2010–2021. Removed entities in the top 15% quantile of missingness (nan_percentage >= 85th percentile).
   - Indicator Removal: Evaluated overall feature null ratios; dropped indicators with >= 50% missing values (e.g., multidimensional poverty headcount ratio).
   - Negative Value Correction: Replaced invalid negative values in percentage features (such as fossil f

In [15]:
# Add all generated documents together
all_rag_documents = []
all_rag_documents.extend(country_docs)        # Country Profiles
all_rag_documents.extend(cluster_documents)    # Cluster Summaries
all_rag_documents.append(glossary_doc)         # Feature Glossary
all_rag_documents.append(methodology_doc)      # Pipeline & Methodology

print(f"Total documents prepared for RAG: {len(all_rag_documents)}")

Total documents prepared for RAG: 187


In [16]:
import json

# Export all generated documents to a JSON file
with open(r"../../docs/all_rag_documents.json", "w", encoding="utf-8") as f:
    json.dump(all_rag_documents, f, indent=4)